Creating publication 

In [0]:

from pathlib import Path
import hashlib
from pyspark.sql import functions as F

# This task must execute as part of a Databricks Job
dbutils.widgets.text("job_run_id", "")
job_run_id = dbutils.widgets.get("job_run_id").strip()

assert job_run_id.isdigit(), "Run this notebook through the Job"

# Identify the exact source file
source_path = Path(
    "/Volumes/workspace/default/raw_landing/yellow_tripdata_2025-01.parquet"
)

sha256 = hashlib.sha256()

with source_path.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        sha256.update(chunk)

file_hash = sha256.hexdigest()
file_bytes = source_path.stat().st_size

# Confirm exactly one manifest record exists for this version
manifest_name = "workspace.control.source_file_manifest"

matching = spark.table(manifest_name).filter(
    (F.col("source_type") == "yellow_taxi") &
    (F.col("source_month") == "2025-01") &
    (F.col("content_sha256") == file_hash)
)

records = matching.collect()

assert len(records) == 1, "Expected one registered source version"
assert records[0]["file_bytes"] == file_bytes
assert records[0]["processing_status"] in ("OBSERVED", "PUBLISHED")

# Add an audit column for the publishing Job run
if "published_run_id" not in spark.table(manifest_name).columns:
    spark.sql(f"""
        ALTER TABLE {manifest_name}
        ADD COLUMNS (published_run_id STRING)
    """)

# Only transition an observed version to published once
if records[0]["processing_status"] == "OBSERVED":
    spark.sql(f"""
        UPDATE {manifest_name}
        SET processing_status = 'PUBLISHED',
            published_at = current_timestamp(),
            published_run_id = '{job_run_id}'
        WHERE source_type = 'yellow_taxi'
          AND source_month = '2025-01'
          AND content_sha256 = '{file_hash}'
          AND processing_status = 'OBSERVED'
    """)
    print("Source version marked PUBLISHED")
else:
    print("Source version already marked PUBLISHED")

# Verify the final state
published = spark.table(manifest_name).filter(
    (F.col("source_type") == "yellow_taxi") &
    (F.col("source_month") == "2025-01") &
    (F.col("content_sha256") == file_hash)
)

assert published.count() == 1
assert published.first()["processing_status"] == "PUBLISHED"

print("PASS: Publication status verified")
display(published)
